<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase6_8_Sweep_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phases 6-8 (Gate 1): ProteinMPNN, Boltz-2 binding test, comparison with the native complex
For the MDM2/p53 gate: all four arms (`gnn`, `oracle`, `baseline`, `scrambled`).

1. **ProteinMPNN** designs binder sequences (target chain fixed, binder chain = the last chain).
2. **Boltz-2** predicts target + designed binder. This is the only real "does it bind" evidence before MD.
3. **Controls:** the native partner with the same target (does Boltz reproduce the real complex?) and **shuffled** binder sequences (what does a non-binder look like?).
4. **Native comparison** per design: contacted target residues vs the native interface (precision / recall / F1), binder-position distance to the native partner, and RFdiffusion-vs-Boltz pose consistency.

**Ranking uses only Boltz confidence.** The native comparison is never used to pick designs. Gate criteria are written in Cell 9 *before* the results are computed.
Expected time on an A100: ProteinMPNN ~10 min, Boltz-2 ~1-2 h for the default 30 designs per arm.

In [1]:
# ── Cell 1: Mount, settings ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, re, random
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive/Protein_interactions'
SPEC_DIR, RFD_DRIVE, GOLD = f'{BASE}/design_specs', f'{BASE}/rfdiff_outputs', f'{BASE}/gold'
G1 = f'{BASE}/gate1'; os.makedirs(G1, exist_ok=True)
CK = f'{G1}/boltz_checkpoints'; os.makedirs(CK, exist_ok=True)
LOC = '/content/gate1'; os.makedirs(LOC, exist_ok=True)

TARGET_FILTER = None   # None = all targets present in rfdiff_outputs/
ARMS          = ['gnn', 'oracle', 'baseline', 'scrambled']
N_BACKBONES   = 15      # backbones per arm taken forward
SEQS_PER_BB   = 1       # best ProteinMPNN sequences kept per backbone
MPNN_SEQS, MPNN_TEMP = 8, 0.1
N_SHUFFLE     = 10      # shuffled-sequence negative controls
SEED = 42; random.seed(SEED); np.random.seed(SEED)

specs = [json.load(open(p)) for p in sorted(glob.glob(f'{SPEC_DIR}/*.json'))]
specs = [s for s in specs if (TARGET_FILTER is None or s['target'] in TARGET_FILTER) and s['arm'] in ARMS]
GINFO = {s['name']: json.load(open(f"{GOLD}/{s['id']}_{s['pdb_id']}/interface_residues.json")) for s in specs}
for s in specs:
    gi = GINFO[s['name']]
    gapped = {s['target'] for s in specs if any(GINFO[s['name']]['target_gaps'].values())}
if gapped:
    print(f"NOTE: {len(gapped)} targets have numbering gaps (missing density): {sorted(gapped)}")
    print("      Boltz models these as one continuous chain. Residue order/count still match the gold")
    print("      target, so the index mapping in Cell 8 is valid; check 'rmsd_target_fit' afterwards -")
    print("      a large value means Boltz's target deviates from the crystal and that target's")
    print("      interface F1 / pose numbers deserve less weight.")
print(len(specs), 'specs;', {s['name']: len(glob.glob(f"{RFD_DRIVE}/{s['name']}/design_*.pdb")) for s in specs}, 'backbones on Drive')
print(len(specs), 'specs;', {s['name']: len(glob.glob(f"{RFD_DRIVE}/{s['name']}/design_*.pdb")) for s in specs}, 'backbones on Drive')

Mounted at /content/drive
NOTE: 8 targets have numbering gaps (missing density): ['BCL2', 'BCL2L1 (Bcl-xL)', 'CDK2', 'CTNNB1', 'HRAS', 'MEN1', 'PCNA', 'RAD51']
      Boltz models these as one continuous chain. Residue order/count still match the gold
      target, so the index mapping in Cell 8 is valid; check 'rmsd_target_fit' afterwards -
      a large value means Boltz's target deviates from the crystal and that target's
      interface F1 / pose numbers deserve less weight.
80 specs; {'10_2XA0_baseline': 15, '10_2XA0_gnn': 15, '10_2XA0_oracle': 15, '10_2XA0_scrambled': 15, '11_1G5J_baseline': 15, '11_1G5J_gnn': 15, '11_1G5J_oracle': 15, '11_1G5J_scrambled': 15, '12_3EZQ_baseline': 15, '12_3EZQ_gnn': 15, '12_3EZQ_oracle': 15, '12_3EZQ_scrambled': 15, '13_1NW9_baseline': 15, '13_1NW9_gnn': 15, '13_1NW9_oracle': 15, '13_1NW9_scrambled': 15, '20_4G0N_baseline': 15, '20_4G0N_gnn': 15, '20_4G0N_oracle': 15, '20_4G0N_scrambled': 15, '22_1KJY_baseline': 15, '22_1KJY_gnn': 15, '22_1KJY_orac

In [2]:
# ── Cell 2: Helper functions (tested offline) ─────────────────────────────────
import re, random
import numpy as np


def kabsch(P, Q):
    # rotation R and translation t so that (P @ R.T + t) best matches Q (least squares)
    Pc, Qc = P.mean(0), Q.mean(0)
    H = (P - Pc).T @ (Q - Qc)
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return R, Qc - Pc @ R.T


def apply_rt(X, R, t):
    return X @ R.T + t


def rmsd(A, B):
    return float(np.sqrt(((A - B) ** 2).sum(1).mean()))


def contact_residues(res_atoms, binder_xyz, cutoff=5.0):
    # res_atoms: list (per target residue) of [n_atoms,3] arrays; returns indices with any atom < cutoff of any binder atom
    from scipy.spatial import cKDTree
    tree = cKDTree(binder_xyz)
    return {i for i, xyz in enumerate(res_atoms) if len(xyz) and tree.query(xyz)[0].min() < cutoff}


def prf(pred, native):
    pred, native = set(pred), set(native)
    tp = len(pred & native)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(native) if native else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def interface_pae(pae, target_len, binder_len):
    # tokens are ordered target chains then binder (the order the chains are written in the YAML)
    pae = np.squeeze(np.asarray(pae))
    if pae.shape[0] != target_len + binder_len: return None
    a = pae[:target_len, target_len:]; b = pae[target_len:, :target_len]
    return float((a.mean() + b.mean()) / 2)


def parse_mpnn_fasta(text, binder_last=True):
    # ProteinMPNN fasta: first record = input sequence, later records = designs with 'score=' in the header.
    # Chains are joined with '/'; the binder is the LAST chain (RFdiffusion writes it last).
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    recs = [(lines[i][1:], lines[i + 1]) for i in range(0, len(lines) - 1, 2)]
    out = []
    for h, seq in recs[1:]:
        m = re.search(r'(?<![A-Za-z_])score=([\d.]+)', h)
        if not m: continue
        out.append((float(m.group(1)), seq.split('/')[-1] if binder_last else seq))
    return out


def low_complexity(seq, max_frac=0.35):
    return max(seq.count(a) for a in set(seq)) / len(seq) > max_frac


def shuffled(seq, seed):
    s = list(seq); random.Random(seed).shuffle(s); return ''.join(s)


def boltz_yaml(chains):
    # chains: list of (id, sequence, msa_mode) with msa_mode in {'server','empty'}
    lines = ['version: 1', 'sequences:']
    for cid, seq, msa in chains:
        lines += ['  - protein:', f'      id: {cid}', f'      sequence: {seq}']
        if msa == 'empty': lines.append('      msa: empty')
    return '\n'.join(lines) + '\n'

_P = np.random.default_rng(0).normal(size=(30, 3)); _R, _t = kabsch(_P, _P @ np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1.]]).T + 1.0)
assert rmsd(apply_rt(_P, _R, _t), _P @ np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1.]]).T + 1.0) < 1e-8
print('Helpers ready.')

Helpers ready.


In [3]:
# ── Cell 3: Collect backbones and identify the binder chain ───────────────────
def chain_layout(pdb_path):
    ch = {}
    for l in open(pdb_path):
        if l.startswith('ATOM') and l[12:16].strip() == 'CA': ch.setdefault(l[21], []).append(int(l[22:26]))
    return ch

BB = {}
for s in specs:
    files = sorted(glob.glob(f"{RFD_DRIVE}/{s['name']}/design_*.pdb"),
                   key=lambda p: int(re.findall(r'design_(\d+)', p)[0]))[:N_BACKBONES]
    d = f"{LOC}/backbones/{s['name']}"; os.makedirs(d, exist_ok=True)
    for f in files:
        shutil.copy2(f, d)
        if os.path.exists(f.replace('.pdb', '.trb')): shutil.copy2(f.replace('.pdb', '.trb'), d)
    lay = chain_layout(files[0]) if files else {}
    # The TARGET chains are exactly the ones named in this spec's contig; the binder is the remaining
    # chain. (Do NOT use "numbered from 1" - some targets, e.g. Bcl-xL/1G5J, are also numbered from 1,
    # and do NOT use file position - RFdiffusion writes the binder first.)
    tgt_chains = {seg[0] for seg in s['segments']}
    cand = [c for c in lay if c not in tgt_chains]
    assert len(cand) == 1, (f"{s['name']}: expected exactly one non-target chain, got {cand}. "
                            f"contig chains={sorted(tgt_chains)}, layout="
                            f"{{{', '.join(f'{c}:{min(v)}-{max(v)}' for c, v in lay.items())}}}")
    bc = cand[0]
    lo, hi = s['binder_len_range']
    if not (lo - 10 <= len(lay[bc]) <= hi + 10):
        print(f"  WARNING {s['name']}: binder chain {bc} has {len(lay[bc])} res, expected ~{lo}-{hi}")
    BB[s['name']] = dict(dir=d, files=[f"{d}/{os.path.basename(f)}" for f in files], binder_chain=bc,
                         target_chains=sorted(tgt_chains),
                         layout={c: (len(v), min(v), max(v)) for c, v in lay.items()})
    print(f"{s['name']:<22} {len(files):3d} backbones | "
          f"chains {{{', '.join(f'{c}:{len(v)}({min(v)}-{max(v)})' for c, v in lay.items())}}} | binder = {bc}")
assert all(b['files'] for b in BB.values()), 'missing backbones - run the Phase 5 notebook first'

10_2XA0_baseline        15 backbones | chains {B:47(1-47), A:137(10-206)} | binder = B
10_2XA0_gnn             15 backbones | chains {B:47(1-47), A:137(10-206)} | binder = B
10_2XA0_oracle          15 backbones | chains {B:54(1-54), A:137(10-206)} | binder = B
10_2XA0_scrambled       15 backbones | chains {B:42(1-42), A:137(10-206)} | binder = B
11_1G5J_baseline        15 backbones | chains {B:47(1-47), A:175(1-215)} | binder = B
11_1G5J_gnn             15 backbones | chains {B:52(1-52), A:175(1-215)} | binder = B
11_1G5J_oracle          15 backbones | chains {B:52(1-52), A:175(1-215)} | binder = B
11_1G5J_scrambled       15 backbones | chains {B:55(1-55), A:175(1-215)} | binder = B
12_3EZQ_baseline        15 backbones | chains {A:107(1-107), I:115(223-337)} | binder = A
12_3EZQ_gnn             15 backbones | chains {A:97(1-97), I:115(223-337)} | binder = A
12_3EZQ_oracle          15 backbones | chains {A:106(1-106), I:115(223-337)} | binder = A
12_3EZQ_scrambled       15 backbones | c

In [4]:
SEL = pd.read_csv(f'{G1}/selected_designs.csv')
print(len(SEL), 'sequences reloaded;', SEL.groupby('arm').size().to_dict())

1200 sequences reloaded; {'baseline': 300, 'gnn': 300, 'oracle': 300, 'scrambled': 300}


In [7]:
# ── Cell 4: ProteinMPNN (binder only; target fixed) ────────────────────────────
MPNN = '/content/ProteinMPNN'
if not os.path.exists(MPNN):
    subprocess.run(f'git clone -q https://github.com/dauparas/ProteinMPNN.git {MPNN}', shell=True)
subprocess.run(f'{sys.executable} -m pip install -q biopython', shell=True)
env = dict(os.environ, PYTHONPATH=MPNN)

rows = []
for s in specs:
    b = BB[s['name']]; out = f"{LOC}/mpnn/{s['name']}"; os.makedirs(out, exist_ok=True)
    parsed, assigned = f'{out}/parsed.jsonl', f'{out}/assigned.jsonl'
    cmds = [f"{sys.executable} {MPNN}/helper_scripts/parse_multiple_chains.py --input_path {b['dir']} --output_path {parsed}",
            f"{sys.executable} {MPNN}/helper_scripts/assign_fixed_chains.py --input_path {parsed} --output_path {assigned} --chain_list \"{b['binder_chain']}\"",
            f"{sys.executable} {MPNN}/protein_mpnn_run.py --jsonl_path {parsed} --chain_id_jsonl {assigned} --out_folder {out} "
            f"--num_seq_per_target {MPNN_SEQS} --sampling_temp {MPNN_TEMP} --seed {SEED} --batch_size 1 --model_name v_48_020 "
            f"--path_to_model_weights {MPNN}/vanilla_model_weights --save_score 1"]
    for c in cmds:
        r = subprocess.run(c, shell=True, capture_output=True, text=True, cwd=MPNN, env=env)
        if r.returncode: raise RuntimeError(f"{s['name']}: {c[:80]}...\n{(r.stderr or r.stdout)[-1500:]}")

    n_kept = 0
    for f in b['files']:
        stem = os.path.splitext(os.path.basename(f))[0]
        recs = sorted(parse_mpnn_fasta(open(f'{out}/seqs/{stem}.fa').read()))
        if not recs: continue
        # Prefer a non-degenerate sequence, but DO NOT discard the backbone if all 8 are degenerate.
        # Record max single-AA fraction as a column and filter at analysis time instead - otherwise n
        # varies 4-15 across arms purely from ProteinMPNN's low-T residue bias, which wrecks the medians.
        clean = [r for r in recs if not low_complexity(r[1])]
        pick  = (clean or recs)[:SEQS_PER_BB]
        for rank, (sc, q) in enumerate(pick):
            maxfrac = max(q.count(a) for a in set(q)) / len(q)
            rows.append(dict(spec=s['name'], arm=s['arm'], backbone=f, stem=stem, rank=rank,
                             mpnn_score=sc, seq=q, binder_len=len(q),
                             max_aa_frac=round(maxfrac, 3), degenerate=bool(maxfrac > 0.35)))
            n_kept += 1
    print(f"{s['name']:<22} designed {n_kept} sequences")

SEL = pd.DataFrame(rows); SEL['run_id'] = SEL.spec + '__' + SEL.stem + '_s' + SEL['rank'].astype(str)
SEL.to_csv(f'{G1}/selected_designs.csv', index=False)
print(SEL.groupby('arm').agg(n=('seq', 'size'), mpnn_score=('mpnn_score', 'median'),
                             length=('binder_len', 'median'),
                             degenerate_frac=('degenerate', 'mean')).round(2).to_string())

10_2XA0_baseline       designed 15 sequences
10_2XA0_gnn            designed 15 sequences
10_2XA0_oracle         designed 15 sequences
10_2XA0_scrambled      designed 15 sequences
11_1G5J_baseline       designed 15 sequences
11_1G5J_gnn            designed 15 sequences
11_1G5J_oracle         designed 15 sequences
11_1G5J_scrambled      designed 15 sequences
12_3EZQ_baseline       designed 15 sequences
12_3EZQ_gnn            designed 15 sequences
12_3EZQ_oracle         designed 15 sequences
12_3EZQ_scrambled      designed 15 sequences
13_1NW9_baseline       designed 15 sequences
13_1NW9_gnn            designed 15 sequences
13_1NW9_oracle         designed 15 sequences
13_1NW9_scrambled      designed 15 sequences
20_4G0N_baseline       designed 15 sequences
20_4G0N_gnn            designed 15 sequences
20_4G0N_oracle         designed 15 sequences
20_4G0N_scrambled      designed 15 sequences
22_1KJY_baseline       designed 15 sequences
22_1KJY_gnn            designed 15 sequences
22_1KJY_or

In [6]:
# ── Cell 5: Boltz-2 in an isolated Python 3.10 venv built with uv ─────────────
# Colab's kernel is 3.13, which has no wheels for dm-tree or the pinned scipy, so pip tries to
# compile them and fails. uv fetches a standalone 3.10 and builds the venv from that instead.
import subprocess, os
BOLTZ_VENV = '/content/boltz_env'; BOLTZ_PY, BOLTZ_BIN = f'{BOLTZ_VENV}/bin/python', f'{BOLTZ_VENV}/bin/boltz'
subprocess.run('pip install -q uv', shell=True)
UV = subprocess.run('which uv', shell=True, capture_output=True, text=True).stdout.strip() or f'{os.path.expanduser("~")}/.local/bin/uv'
subprocess.run(f'{UV} python install 3.10', shell=True, capture_output=True)

def run(cmd, label=''):
    print(f'  {label or cmd[:70]}...', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print('OK' if r.returncode == 0 else 'FAILED'); (r.returncode and print(r.stderr[-1800:])); return r.returncode

if os.path.exists(BOLTZ_VENV): run(f'rm -rf {BOLTZ_VENV}', 'removing old 3.13 venv')
run(f'{UV} venv --python 3.10 {BOLTZ_VENV}', 'creating 3.10 venv (uv)')
print('  venv python:', subprocess.run(f'{BOLTZ_PY} --version', shell=True, capture_output=True, text=True).stdout.strip())

open(f'{BOLTZ_VENV}/constraints.txt', 'w').write('numpy>=1.26,<2.0\nscipy==1.13.1\n')
C = f'--constraint {BOLTZ_VENV}/constraints.txt'
run(f'{UV} pip install --python {BOLTZ_PY} {C} "numpy>=1.26,<2.0" "scipy==1.13.1"', 'numpy<2 + scipy pin')
run(f'{UV} pip install --python {BOLTZ_PY} {C} boltz', 'boltz')
run(f'{UV} pip install --python {BOLTZ_PY} {C} torch --index-url https://download.pytorch.org/whl/cu121', 'torch (cu121)')
h = subprocess.run(f'{BOLTZ_BIN} predict --help', shell=True, capture_output=True, text=True)
print('boltz CLI:', 'OK' if h.returncode == 0 else 'FAILED\n' + h.stderr[-800:])

  removing old 3.13 venv... OK
  creating 3.10 venv (uv)... OK
  venv python: Python 3.10.22
  numpy<2 + scipy pin... OK
  boltz... OK
  torch (cu121)... OK
boltz CLI: OK


In [7]:
# ── Cell 6: Build Boltz-2 inputs - designs + positive control (native) + negative controls (shuffled) ──
def target_chains(spec_name):
    gi = GINFO[spec_name]; ids = list(gi['target_seq'].keys())              # file order = residue order in gold target.pdb
    return [(chr(65 + i), gi['target_seq'][c]) for i, c in enumerate(ids)], ids
YDIR = f'{LOC}/boltz_yaml'; shutil.rmtree(YDIR, ignore_errors=True); os.makedirs(YDIR)
RUNS = {}                                                                      # run_id -> dict(spec, kind, binder_seq)
def add_run(run_id, spec_name, kind, bseq):
    tch, _ = target_chains(spec_name); bl = chr(65 + len(tch))
    open(f'{YDIR}/{run_id}.yaml', 'w').write(boltz_yaml([(c, q, 'server') for c, q in tch] + [(bl, bseq, 'empty')]))
    RUNS[run_id] = dict(spec=spec_name, kind=kind, binder_seq=bseq, binder_chain=bl, n_target=sum(len(q) for _, q in tch))

SPEC_TARGET = {sp['name']: sp['target'] for sp in specs}
RUN_TARGET = {rid: SPEC_TARGET.get(sp) for rid, sp in zip(SEL.run_id, SEL.spec)}
for _, r in SEL.iterrows(): add_run(r.run_id, r.spec, 'design', r.seq)
# one native-partner positive control PER TARGET (first spec seen for that target)
seen_t = set()
for s in specs:
    if s['target'] in seen_t: continue
    seen_t.add(s['target'])
    gi = GINFO[s['name']]; ps = list(gi['partner_seq'].values())
    if len(ps) == 1: add_run(f"ctrl_native__{s['id']}_{s['target']}", s['name'], 'native', ps[0])
    else: print(f"{s['target']}: multi-chain partner, native control skipped")
# shuffled negative controls PER TARGET: N_SHUFFLE_PER drawn from that target's gnn/oracle designs
N_SHUFFLE_PER = 3
for tgt in sorted(seen_t):
    sub = SEL[(SEL.arm.isin(['gnn','oracle'])) & (SEL.run_id.map(lambda x: RUN_TARGET.get(x)) == tgt)]
    if len(sub) == 0: sub = SEL[SEL.arm.isin(['gnn','oracle'])]
    sub = sub.sample(min(N_SHUFFLE_PER, len(sub)), random_state=SEED)
    for i, (_, r) in enumerate(sub.iterrows()):
        add_run(f'ctrl_shuffle{i}__{tgt}', r.spec, 'shuffled', shuffled(r.seq, SEED + i))
print(len(RUNS), 'Boltz inputs:', pd.Series([v['kind'] for v in RUNS.values()]).value_counts().to_dict())
json.dump(RUNS, open(f'{G1}/boltz_runs.json', 'w'), indent=1)

1280 Boltz inputs: {'design': 1200, 'shuffled': 60, 'native': 20}


In [12]:
import pandas as pd
sz = pd.DataFrame([{'run_id': r, 'n_target': RUNS[r]['n_target'],
                    'total': RUNS[r]['n_target'] + len(RUNS[r]['binder_seq']),
                    'spec': RUNS[r]['spec']} for r in pending])
sz['batch'] = range(len(sz)); sz['batch'] //= 20
print(sz.groupby('batch').agg(spec=('spec','first'), median_len=('total','median')).head(20).to_string())
print('\nlargest targets still queued:')
print(sz.groupby('spec').total.median().sort_values(ascending=False).head(8).to_string())

                   spec  median_len
batch                              
0      10_2XA0_baseline       186.0
1           10_2XA0_gnn       188.5
2        10_2XA0_oracle       189.0
3      11_1G5J_baseline       229.0
4           11_1G5J_gnn       228.0
5        11_1G5J_oracle       225.0
6      12_3EZQ_baseline       215.0
7           12_3EZQ_gnn       210.0
8        12_3EZQ_oracle       213.5
9      13_1NW9_baseline       187.0
10          13_1NW9_gnn       194.5
11       13_1NW9_oracle       188.0
12     20_4G0N_baseline       244.0
13          20_4G0N_gnn       242.5
14       20_4G0N_oracle       251.0
15     22_1KJY_baseline       369.0
16          22_1KJY_gnn       372.0
17       22_1KJY_oracle       370.5
18     24_1J2J_baseline       213.0
19          24_1J2J_gnn       213.0

largest targets still queued:
spec
4_1JSU_baseline      621.5
4_1JSU_oracle        612.0
4_1JSU_scrambled     605.0
4_1JSU_gnn           605.0
37_1JDH_scrambled    559.0
37_1JDH_oracle       558.5
37_1JDH_ba

In [13]:
# ── Cell 7 (v2): full local clean each batch + timing split so we can see where the time goes ──
def done(run_id): return bool(glob.glob(f'{CK}/{run_id}/*confidence*.json'))
pending = [r for r in RUNS if not done(r)]
print(f'{len(RUNS) - len(pending)} already done, {len(pending)} to run')
BATCH, t_all = 20, time.time()
for b0 in range(0, len(pending), BATCH):
    batch = pending[b0:b0 + BATCH]
    bdir = f'{LOC}/boltz_batch'; shutil.rmtree(bdir, ignore_errors=True); os.makedirs(bdir)
    for r in batch: shutil.copy2(f'{YDIR}/{r}.yaml', bdir)
    out = f'{LOC}/boltz_out'; shutil.rmtree(out, ignore_errors=True)   # FULL clean - nothing accumulates

    t0 = time.time()
    p = subprocess.run(f'{BOLTZ_BIN} predict {bdir} --out_dir {out} --use_msa_server --override --no_kernels',
                       shell=True, capture_output=True, text=True)
    t_pred = time.time() - t0

    t1 = time.time(); got = 0
    for r in batch:
        if not glob.glob(f'{out}/**/*confidence*{r}_model_0.json', recursive=True): continue
        os.makedirs(f'{CK}/{r}', exist_ok=True)
        for pat in (f'*confidence*{r}_model_0.json', f'{r}_model_0.cif', f'pae_{r}_model_0.npz'):
            for f in glob.glob(f'{out}/**/{pat}', recursive=True): shutil.copy2(f, f'{CK}/{r}')
        got += 1
    t_copy = time.time() - t1

    bn = b0 // BATCH + 1
    print(f'batch {bn}/{-(-len(pending)//BATCH)}: {got}/{len(batch)} ok | '
          f'predict {t_pred/60:.1f} min | copy-to-Drive {t_copy/60:.1f} min (rc={p.returncode})', flush=True)
    if p.returncode and not got: print(p.stderr[-2500:]); break
print(f'total {(time.time() - t_all)/60:.1f} min;', sum(done(r) for r in RUNS), '/', len(RUNS), 'complete')

187 already done, 1093 to run
batch 1/55: 20/20 ok | predict 5.4 min | copy-to-Drive 0.0 min (rc=0)
batch 2/55: 20/20 ok | predict 5.6 min | copy-to-Drive 0.0 min (rc=0)
batch 3/55: 20/20 ok | predict 5.6 min | copy-to-Drive 0.0 min (rc=0)
batch 4/55: 20/20 ok | predict 5.6 min | copy-to-Drive 0.0 min (rc=0)
batch 5/55: 20/20 ok | predict 5.6 min | copy-to-Drive 0.0 min (rc=0)
batch 6/55: 20/20 ok | predict 6.0 min | copy-to-Drive 0.0 min (rc=0)
batch 7/55: 20/20 ok | predict 6.1 min | copy-to-Drive 0.0 min (rc=0)
batch 8/55: 20/20 ok | predict 6.1 min | copy-to-Drive 0.0 min (rc=0)
batch 9/55: 20/20 ok | predict 7.6 min | copy-to-Drive 0.0 min (rc=0)
batch 10/55: 20/20 ok | predict 7.6 min | copy-to-Drive 0.0 min (rc=0)
batch 11/55: 20/20 ok | predict 7.6 min | copy-to-Drive 0.0 min (rc=0)
batch 12/55: 20/20 ok | predict 5.7 min | copy-to-Drive 0.0 min (rc=0)
batch 13/55: 20/20 ok | predict 5.7 min | copy-to-Drive 0.0 min (rc=0)
batch 14/55: 20/20 ok | predict 5.7 min | copy-to-Drive 

In [17]:
# ── Cell 8: Score every run - Boltz confidence, interface PAE, native comparison, RFdiffusion consistency ──
from Bio import PDB
S = PDB.MMCIFParser(QUIET=True); P = PDB.PDBParser(QUIET=True)

def residues_of(chain): return [r for r in chain if r.id[0] == ' ']
def heavy(r): return np.array([a.coord for a in r if (a.element or '').strip() != 'H'])
def ca(r): return r['CA'].coord if 'CA' in r else heavy(r)[0]
def rkey(r): return f"{r.get_parent().id}:{r.id[1]}{r.id[2].strip()}"

NATIVE = {}
for s in specs:
    d = f"{GOLD}/{s['id']}_{s['pdb_id']}"; gi = GINFO[s['name']]
    _t = [r for c in P.get_structure('t', f'{d}/target.pdb')[0] for r in residues_of(c)]
    _p = [r for c in P.get_structure('p', f'{d}/partner.pdb')[0] for r in residues_of(c)]
    NATIVE[s['name']] = dict(t_ca=np.array([ca(r) for r in _t]),
                             p_com=np.array([ca(r) for r in _p]).mean(0),
                             keys=gi['target_residues'], contact=set(gi['interface_contact_5A']), n=len(_t))
    assert len(_t) == len(gi['target_residues']), f"{s['name']}: gold target.pdb vs interface_residues.json mismatch"
sel_idx = SEL.set_index('run_id')

rows = []
for run_id, meta in RUNS.items():
    if not done(run_id): continue
    ck = f'{CK}/{run_id}'; conf = json.load(open(glob.glob(f'{ck}/*confidence*.json')[0]))
    row = dict(run_id=run_id, kind=meta['kind'], spec=meta['spec'],
               iptm=conf.get('iptm'), ptm=conf.get('ptm'), plddt=conf.get('complex_plddt'))
    pz = glob.glob(f'{ck}/pae_*.npz')
    if pz:
        z = np.load(pz[0])
        row['iface_pae'] = interface_pae(z['pae'] if 'pae' in z.files else z[z.files[0]],
                                         meta['n_target'], len(meta['binder_seq']))
    nat = NATIVE[meta['spec']]
    try:
        m = S.get_structure('x', glob.glob(f'{ck}/*_model_0.cif')[0])[0]
        tch  = [c for c in m if c.id != meta['binder_chain']]
        tres = [r for c in tch for r in residues_of(c)]
        bres = residues_of(m[meta['binder_chain']])
        assert len(tres) == nat['n'], f"target residue count {len(tres)} != gold {nat['n']}"

        # --- native comparison: superpose Boltz target onto the gold target, compare binder placement ---
        R, t = kabsch(np.array([ca(r) for r in tres]), nat['t_ca'])
        b_ca = apply_rt(np.array([ca(r) for r in bres]), R, t)
        cont = contact_residues([heavy(r) for r in tres], np.vstack([heavy(r) for r in bres]), 5.0)
        p_, r_, f1 = prf({nat['keys'][i] for i in cont}, nat['contact'])
        row.update(prec=p_, rec=r_, f1=f1, n_contact=len(cont),
                   com_dist=float(np.linalg.norm(b_ca.mean(0) - nat['p_com'])),
                   rmsd_target_fit=rmsd(apply_rt(np.array([ca(r) for r in tres]), R, t), nat['t_ca']))

        # --- pose self-consistency: does Boltz put the binder where RFdiffusion did? ---
        if meta['kind'] == 'design' and run_id in sel_idx.index:
            bb_model = P.get_structure('b', sel_idx.loc[run_id, 'backbone'])[0]
            bc  = BB[meta['spec']]['binder_chain']          # binder = chain numbered from 1 (see Cell 3)
            rb  = residues_of(bb_model[bc])
            rt_ = [r for c in bb_model if c.id != bc for r in residues_of(c)]
            # RFdiffusion's target is the CROPPED contig (original numbering); Boltz's target is the FULL
            # gold target (renumbered 1..N). Match by gold residue key -> ordinal position, so cropped
            # targets superpose correctly. Without this, pose_rmsd was silently NaN for the 8 cropped targets.
            k2i  = {k: i for i, k in enumerate(nat['keys'])}
            idxs = [k2i[rkey(r)] for r in rt_ if rkey(r) in k2i]
            if len(idxs) == len(rt_) and len(rb) == len(bres) and len(idxs) >= 10:
                rf_ca = np.array([ca(r) for r in rt_])
                bz_ca = np.array([ca(tres[i]) for i in idxs])      # same residues, Boltz coordinates
                R2, t2 = kabsch(rf_ca, bz_ca)
                rb_ca  = apply_rt(np.array([ca(r) for r in rb]), R2, t2)
                bb_ca  = np.array([ca(r) for r in bres])
                row['pose_rmsd'] = rmsd(rb_ca, bb_ca)
                row['pose_fit_n'] = len(idxs)
                R3, t3 = kabsch(rb_ca, bb_ca)
                row['binder_fold_rmsd'] = rmsd(apply_rt(rb_ca, R3, t3), bb_ca)
            else:
                row['pose_note'] = f'unmatched rf_target={len(rt_)} matched={len(idxs)} rb={len(rb)} bres={len(bres)}'
    except Exception as e:
        row['error'] = str(e)[:160]
    rows.append(row)

RES = pd.DataFrame(rows).merge(SEL[['run_id', 'arm', 'mpnn_score']], on='run_id', how='left')
RES.loc[RES.kind != 'design', 'arm'] = RES.kind
for c_ in ('iptm', 'iface_pae', 'f1', 'com_dist', 'pose_rmsd', 'binder_fold_rmsd', 'plddt', 'pose_note', 'error'):
    if c_ not in RES: RES[c_] = np.nan
# target label for the cross-target analysis
SPEC_TGT = {sp['name']: sp['target'] for sp in specs}
RES['target'] = RES.spec.map(SPEC_TGT)
RES.to_csv(f'{G1}/gate1_results.csv', index=False)

n_err = int(RES['error'].notna().sum())
print(f"{len(RES)} runs scored; errors: {n_err}")
if n_err: print(RES[RES['error'].notna()][['run_id', 'error']].head(10).to_string(index=False))

des = RES[RES.kind == 'design']
print(f"\npose_rmsd computed for {int(des.pose_rmsd.notna().sum())}/{len(des)} designs "
      f"(this is the metric that separates the arms - it must NOT be mostly NaN)")
bad = des[des.pose_rmsd.isna() & des.pose_note.notna()]
if len(bad):
    print('  unmatched examples:'); print(bad[['run_id', 'pose_note']].head(5).to_string(index=False))

print('\nper-target design counts and pose coverage:')
print(des.groupby('target').agg(n=('run_id', 'size'), pose_ok=('pose_rmsd', lambda x: int(x.notna().sum()))).to_string())

print('\nControls:')
ctl = RES[RES.kind != 'design']
if len(ctl): print(ctl[['run_id', 'kind', 'iptm', 'iface_pae', 'f1', 'com_dist']].round(3).to_string(index=False))

1280 runs scored; errors: 0

pose_rmsd computed for 1200/1200 designs (this is the metric that separates the arms - it must NOT be mostly NaN)

per-target design counts and pose coverage:
                  n  pose_ok
target                      
ARF1             60       60
BCL2             60       60
BCL2L1 (Bcl-xL)  60       60
BRCA1            60       60
CDK2             60       60
CREBBP (KIX)     60       60
CTNNB1           60       60
FAS              60       60
GNAI1            60       60
HRAS             60       60
JUN              60       60
KEAP1            60       60
MDM2             60       60
MEN1             60       60
MYC              60       60
PCNA             60       60
RAD51            60       60
VCL              60       60
VHL              60       60
XIAP (BIR3)      60       60

Controls:
                         run_id     kind  iptm  iface_pae    f1  com_dist
           ctrl_native__10_BCL2   native 0.965      3.228 0.967     0.610
ctrl_native__11

In [18]:
RES[RES.kind=='design'].groupby('target').rmsd_target_fit.median().sort_values(ascending=False).round(2)

,rmsd_target_fit
target,
FAS,11.79
BCL2,7.07
MEN1,3.22
CREBBP (KIX),2.84
BCL2L1 (Bcl-xL),2.82
CTNNB1,2.50
JUN,2.42
BRCA1,2.26
MYC,2.10


In [19]:
import sys; sys.path.insert(0, BASE)  # pipeline_metrics.py lives at the repo root
from pipeline_metrics import COM_DIST_MAX
nat = RES[RES.kind == 'native'].set_index('target')
fit = RES[RES.kind == 'design'].groupby('target').rmsd_target_fit.median()
q = nat[['f1', 'com_dist', 'iptm']].join(fit.rename('target_fit'))
# 'trusted' is a per-TARGET inclusion gate: native-control f1/com_dist
# (did Boltz reproduce the native complex) PLUS target_fit, the median
# design-run rmsd_target_fit (does this target's design batch align well
# enough to the gold target to trust it) -- a native-control-plus-design-
# alignment gate, NOT the same thing as Phase9's per-design 'hit'/
# 'self_consistent' outcome (pipeline_metrics.classify_design(), which
# measures pose_rmsd vs. the RFdiffusion backbone, a different quantity
# than target_fit). Reuses COM_DIST_MAX only as a shared distance
# convention for the native-control com_dist check.
# claiming to be the same metric as pipeline_metrics.classify_design().
q['trusted'] = (q.f1 >= 0.70) & (q.com_dist <= COM_DIST_MAX) & (q.target_fit <= 8)
print(q.round(2).sort_values('trusted').to_string())
print('\nexcluded:', sorted(q.index[~q.trusted]))
q.to_csv(f'{G1}/target_quality.csv')

                   f1  com_dist  iptm  target_fit  trusted
target                                                    
FAS              0.46     30.61  0.95       11.79    False
XIAP (BIR3)      0.65     35.13  0.55        0.37    False
BCL2             0.97      0.61  0.96        7.07     True
BRCA1            0.82      1.90  0.90        2.26     True
MDM2             0.91      0.77  0.96        0.58     True
CDK2             0.89      4.35  0.89        2.02     True
VCL              0.92      0.18  0.98        1.26     True
CTNNB1           0.89      1.08  0.96        2.50     True
KEAP1            0.95      2.42  0.94        0.35     True
VHL              0.83      2.47  0.95        0.70     True
CREBBP (KIX)     0.94      1.41  0.96        2.84     True
RAD51            0.99      0.67  0.93        1.83     True
MYC              0.99      0.81  0.94        2.10     True
JUN              0.94      4.77  0.92        2.42     True
ARF1             0.81      0.39  0.91        0.55     Tr

In [20]:
# ── Cell 9: Gate 1 summary and verdict ────────────────────────────────────────
# Criteria fixed BEFORE looking at design results:
#  (a) Positive control: Boltz reproduces the native p53/MDM2 interface (native-partner run has F1 >= 0.5).
#  (b) Negative control: shuffled sequences score clearly lower than the passing designs (median iptm).
#  (c) Ceiling: the ORACLE arm yields >= 3 designs with F1 >= 0.5 (the pipeline can make native-like binders at all).
#  (d) Specificity: best-confidence ORACLE design has higher F1 than best-confidence SCRAMBLED design.
des = RES[(RES.kind == 'design') & RES.f1.notna()].copy()
nat_ctrl, shuf = RES[RES.kind == 'native'], RES[RES.kind == 'shuffled']
PASS_IPTM = max(0.5, float(nat_ctrl.iptm.iloc[0]) - 0.2) if len(nat_ctrl) else 0.6
PASS_PAE  = float(nat_ctrl.iface_pae.iloc[0]) + 4 if len(nat_ctrl) and nat_ctrl.iface_pae.notna().all() else 10.0
des['pass'] = (des.iptm >= PASS_IPTM) & (des.iface_pae <= PASS_PAE)
print(f'Pass filter (from the native control): iptm >= {PASS_IPTM:.2f} and interface PAE <= {PASS_PAE:.1f}\n')

def top1(g): return g.sort_values(['iptm', 'iface_pae'], ascending=[False, True]).iloc[0]
rows = []
for arm in ARMS:
    g = des[des.arm == arm]
    if not len(g): continue
    top5 = g.sort_values(['iptm', 'iface_pae'], ascending=[False, True]).head(5)
    rows.append(dict(arm=arm, n=len(g), n_pass=int(g['pass'].sum()), median_iptm=g.iptm.median(), median_pae=g.iface_pae.median(),
                     top1_f1=top1(g).f1, top5_mean_f1=top5.f1.mean(), best_f1=g.f1.max(), n_f1_ge_0_5=int((g.f1 >= 0.5).sum()),
                     top5_com_dist=top5.com_dist.median(), median_pose_rmsd=g.pose_rmsd.median()))
tab = pd.DataFrame(rows).set_index('arm'); print(tab.round(2).to_string())
print('\nControls:'); print(RES[RES.kind.isin(['native', 'shuffled'])].groupby('kind')[['iptm', 'iface_pae', 'f1']].agg(['median', 'max']).round(2).to_string())

ok_a = len(nat_ctrl) > 0 and float(nat_ctrl.f1.iloc[0]) >= 0.5
ok_b = len(shuf) > 0 and float(shuf.iptm.median()) < float(des[des['pass']].iptm.median() if des['pass'].any() else des.iptm.median())
ok_c = 'oracle' in tab.index and int(tab.loc['oracle', 'n_f1_ge_0_5']) >= 3
ok_d = {'oracle', 'scrambled'} <= set(tab.index) and tab.loc['oracle', 'top1_f1'] > tab.loc['scrambled', 'top1_f1']
print('\nGATE 1 CRITERIA')
for lab, ok_ in (('(a) Boltz reproduces the native complex', ok_a), ('(b) shuffled sequences score lower than designs', ok_b),
                 ('(c) oracle arm makes >=3 native-like designs', ok_c), ('(d) oracle top-1 beats scrambled top-1', ok_d)):
    print(f'  {"PASS" if ok_ else "FAIL"}  {lab}')
print('\n=>', 'GATE 1 PASSED - safe to scale to more targets' if all([ok_a, ok_b, ok_c, ok_d]) else
      'GATE 1 NOT PASSED - do not scale yet; send me this output and Cell 8 errors')
print('\nAlso informative: does the GNN arm approach the oracle?  gnn top-1 F1 =', round(float(tab.loc['gnn', 'top1_f1']), 2) if 'gnn' in tab.index else 'n/a',
      '| oracle =', round(float(tab.loc['oracle', 'top1_f1']), 2) if 'oracle' in tab.index else 'n/a')

Pass filter (from the native control): iptm >= 0.76 and interface PAE <= 7.2

             n  n_pass  median_iptm  median_pae  top1_f1  top5_mean_f1  best_f1  n_f1_ge_0_5  top5_com_dist  median_pose_rmsd
arm                                                                                                                          
gnn        300     166         0.86        6.21     0.88          0.70     1.00          189           5.48              7.27
oracle     300     155         0.83        6.92     0.88          0.87     0.98          201           3.92             10.88
baseline   300     115         0.74        9.39     0.42          0.51     0.98          144          11.49             32.51
scrambled  300      92         0.71       10.54     0.00          0.47     0.96          143          11.11             43.55

Controls:
           iptm       iface_pae            f1      
         median   max    median    max median   max
kind                                               